# dscraft.clean quickstart

This notebook demonstrates `dscraft.clean`'s near-duplicate text detection: embed a handful of text rows via a fully hermetic, synthetic ONNX Runtime embedding model (`build_synthetic_embedding_model`), then flag near-duplicate pairs by cosine similarity via `detect_near_duplicate_text`.

No network access or bundled model file is required. See the package README for how a real production sentence-embedding model would be wired in via the same `EmbeddingModel`/`detect_near_duplicate_text` API.

This mirrors `packages/dscraft/examples/clean/dedup_example.py` in notebook form.

Requires the `clean` extra:
```bash
pip install -e "packages/dscraft[clean]"
```

In [1]:
from dscraft.clean import build_synthetic_embedding_model, detect_near_duplicate_text

ROWS = [
    "The quick brown fox jumps over the lazy dog.",
    "The quick brown fox jumps over the lazy dog",  # near-duplicate of row 0 (punctuation only)
    "the QUICK brown fox JUMPS over the lazy dog!!!",  # near-duplicate of row 0 (case/punctuation)
    "Quantum entanglement enables non-local correlations between particles.",  # distinct
    "Sourdough bread requires a long, slow fermentation of the starter.",  # distinct
]

model = build_synthetic_embedding_model(vocab_dim=128, embedding_dim=32)
print(f"Embedding {len(ROWS)} rows via ONNX Runtime (no PyTorch/transformers)...")

embeddings, report = detect_near_duplicate_text(ROWS, model, threshold=0.9)
print(f"Embeddings shape: {embeddings.shape}")
print(
    f"Near-duplicate scan: {len(report.pairs)} pair(s) flagged "
    f"at cosine-similarity threshold {report.threshold}."
)

Embedding 5 rows via ONNX Runtime (no PyTorch/transformers)...


Embeddings shape: (5, 32)
Near-duplicate scan: 3 pair(s) flagged at cosine-similarity threshold 0.9.


## Flagged near-duplicate pairs and distinct rows

In [2]:
for pair in report.pairs:
    print(
        f"  rows [{pair.index_a}] <-> [{pair.index_b}]  "
        f"similarity={pair.similarity:.4f}"
    )
    print(f"    [{pair.index_a}] {ROWS[pair.index_a]!r}")
    print(f"    [{pair.index_b}] {ROWS[pair.index_b]!r}")

flagged = report.flagged_indices()
distinct_rows = [
    i
    for i in range(len(ROWS))
    if i not in flagged and i not in report.zero_vector_row_indices
]
print()
print(f"Rows not flagged as near-duplicates: {distinct_rows}")

  rows [0] <-> [1]  similarity=1.0000
    [0] 'The quick brown fox jumps over the lazy dog.'
    [1] 'The quick brown fox jumps over the lazy dog'
  rows [0] <-> [2]  similarity=1.0000
    [0] 'The quick brown fox jumps over the lazy dog.'
    [2] 'the QUICK brown fox JUMPS over the lazy dog!!!'
  rows [1] <-> [2]  similarity=1.0000
    [1] 'The quick brown fox jumps over the lazy dog'
    [2] 'the QUICK brown fox JUMPS over the lazy dog!!!'

Rows not flagged as near-duplicates: [3, 4]


## Summary

This notebook embedded 5 text rows via `dscraft.clean`'s hermetic ONNX Runtime embedding model and used `detect_near_duplicate_text` to flag near-duplicate pairs by cosine similarity -- correctly identifying the two punctuation/case variants of the same sentence as near-duplicates while leaving the two genuinely distinct sentences unflagged.

See `packages/dscraft/README.md`'s `## dscraft.clean` section for more detail, including the higher-level `Sanitizer` audit-then-clean workflow (DeCoLe label-error detection, contamination auditing, Dataset Integrity Score).